# 18：TabDiffモデルの学習・保存専用

Parquet読込 → 前処理 → 学習 → 通常モデル＋EMAモデル保存。仮想患者データの生成は19で行います。

学習セルには「現在の経過時間」「予想総所要時間」の2項目だけを表示します（5秒ごとに同じ表示を更新）。最初の1エポックが終わるまでは推定中です。予想時間は実測から更新する目安で、保証値ではありません。

モデルだけでなく、`artifacts/18_tabdiff/実行時刻/`の前処理情報と`external/TabDiff/data/データ名/`も保存してください。19での生成に必要です。

公式実装：https://github.com/MinkaiXu/TabDiff
Python・PyTorch・依存パッケージは構築済みのGPU用.venvを使用してください。

修正版：EMAを各バッチ終了後に更新し、通常モデルとEMAモデルを両方保存します。既存モデルに遡って適用されないため、最初から学習してください。


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, subprocess, hashlib, time
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
INPUT = PROJECT / "data/processed/utstein_modified.parquet"
TABDIFF_REPO = PROJECT / "external/TabDiff"  # 既存の公式コードがあれば変更
TABDIFF_PYTHON = Path(sys.executable)       # TabDiff環境のPython
DEFAULT_GENERATE_COUNT = 10_000
EPOCHS = 200                              # 動作確認なら2、本学習は損失を見て調整
TRAIN_BATCH_SIZE = 1024
DEFAULT_SAMPLE_BATCH_SIZE = 512                   # GPUメモリ不足なら下げる
GPU = 0                                  # CPUなら-1（大規模データは非常に時間がかかります）
SEED = 42
EMA_DECAY = 0.997  # 各バッチでEMAを更新

MAX_TRAIN_ROWS = None                     # 全件使用。試運転のみ例：20000
EXCLUDE_COLUMNS = ["ID", "id", "患者ID", "患者番号", "症例ID"]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DATA_NAME = "utstein_" + RUN_ID
EXP_NAME = "unconditional"
OUT = PROJECT / "artifacts/18_tabdiff" / RUN_ID
assert INPUT.is_file(), f"入力がありません: {INPUT}"
assert (TABDIFF_REPO / "main.py").is_file(), f"公式TabDiffのパスを指定してください: {TABDIFF_REPO}"
OUT.mkdir(parents=True, exist_ok=False)
print("入力:", INPUT, "\n学習出力:", OUT)
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024**2), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
# 標準出力・進捗バー・警告はログファイルへ保存します。
# 画面の表示は同じ場所を更新し、ログ行を大量に増やしません。
from collections import deque

def format_duration(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, sec = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{sec:02d}"

def run_command(args, log_name, show_timing=False):
    env = os.environ.copy()
    env.update(WANDB_MODE="disabled", PYTHONUNBUFFERED="1", PYTHONHASHSEED=str(SEED))
    log_path = OUT / log_name
    progress_path = OUT / "training_progress.json"
    if show_timing:
        progress_path.unlink(missing_ok=True)
    started = time.monotonic()
    handle = None
    def update_display(finished=False):
        nonlocal handle
        elapsed = time.monotonic() - started
        estimate = None
        if finished:
            estimate = elapsed
        elif progress_path.is_file():
            try:
                progress = json.loads(progress_path.read_text(encoding="utf-8"))
                done = progress.get("completed_epochs", 0)
                if done > 0:
                    # 起動・前処理時間も含む総経過時間を完了epoch数で外挿。
                    estimate = elapsed + progress["training_elapsed"] / done * max(0, EPOCHS-done)
            except (OSError, ValueError, KeyError):
                pass
        text = ("現在の経過時間：" + format_duration(elapsed)
                + "\n予想総所要時間：" + (format_duration(estimate) if estimate is not None else "推定中（最初の1エポック完了後に表示）"))
        if handle is None:
            handle = display(text, display_id=True)
        else:
            handle.update(text)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(list(map(str, args)), cwd=TABDIFF_REPO, env=env,
                                stdout=log, stderr=subprocess.STDOUT)
        try:
            if show_timing:
                update_display()
            last_update = time.monotonic()
            while proc.poll() is None:
                time.sleep(1)
                if show_timing and time.monotonic()-last_update >= 5:
                    update_display()
                    last_update = time.monotonic()
        except KeyboardInterrupt:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            print(f"中断しました。ログ: {log_path}")
            raise
    if proc.returncode:
        with log_path.open(encoding="utf-8", errors="replace") as f:
            print("".join(deque(f, maxlen=80)))
        raise RuntimeError(f"処理失敗。ログ: {log_path}")
    if show_timing:
        update_display(finished=True)

# GPU環境確認の詳細もenvironment.logへ保存します。
run_command([TABDIFF_PYTHON, "-c",
    "import torch,pandas,numpy,sklearn,sdmetrics,wandb,tomli,tomli_w; "
    "print('torch',torch.__version__);print('CUDA',torch.cuda.is_available()); "
    "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], "environment.log")
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TABDIFF_REPO, text=True).strip()


## データの前処理
全行をランダムに学習90%・ホールドアウト10%へ分割します。数値の中央値は学習側のみから計算します。
数値欠損は中央値で埋め、欠損フラグを追加して欠損パターンも学習します。カテゴリは内部文字コードに変換し、生成後に復元します。日時は1970年からの経過日数として数値化します。IDは学習しません。
全欠損列・定数列はモデルから外し、出力時に復元します。カテゴリが非常に多い列はエラーにして確認を求めます。

In [ ]:
real = pd.read_parquet(INPUT, engine="pyarrow")
if not isinstance(real, pd.DataFrame) or not real.columns.is_unique:
    raise ValueError("入力は列名が一意のDataFrameにしてください。")
real = real.drop(columns=[c for c in EXCLUDE_COLUMNS if c in real.columns]).copy()
if MAX_TRAIN_ROWS is not None:
    real = real.sample(n=min(MAX_TRAIN_ROWS, len(real)), random_state=SEED)
real = real.reset_index(drop=True)
if len(real) < 20:
    raise ValueError("少なくとも20行必要です。")
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(real))
cut = int(len(real) * .9)
train_idx, test_idx = indices[:cut], indices[cut:]
schema, encoded = {}, pd.DataFrame(index=real.index)
constants = {}
num_names, cat_names = [], []
for i, column in enumerate(real.columns):
    series = real[column]
    key = f"f{i:03d}"
    # 日付のみのobject列にも対応
    is_date = pd.api.types.is_datetime64_any_dtype(series) or column in ["発生年月日ウツタイン", "覚知ウツタイン"]
    if series.nunique(dropna=False) <= 1:
        constants[column] = series.iloc[0]
        continue
    spec = {"key": key, "dtype": series.dtype}
    if is_date or (pd.api.types.is_numeric_dtype(series) and not isinstance(series.dtype, pd.CategoricalDtype)):
        if is_date:
            dt = pd.to_datetime(series, errors="coerce")
            values = (dt - pd.Timestamp("1970-01-01")).dt.total_seconds() / 86400
            spec["kind"] = "datetime"
        else:
            values = pd.to_numeric(series, errors="coerce").astype(float)
            spec["kind"] = "numeric"
        values = values.replace([np.inf, -np.inf], np.nan)
        observed = values.loc[train_idx].dropna()
        if observed.empty:
            raise ValueError(f"学習側に有効な数値がありません: {column}")
        spec.update(median=float(observed.median()), minimum=float(observed.min()),
                    maximum=float(observed.max()), integer=bool((observed % 1 == 0).all()))
        encoded[key] = values.fillna(spec["median"])
        num_names.append(key)
        if values.isna().any():
            spec["mask"] = key + "_missing"
            encoded[spec["mask"]] = np.where(values.isna(), "missing_yes", "missing_no")
            cat_names.append(spec["mask"])
    else:
        spec["kind"] = "category"
        labels = series.astype("string")
        unique = labels.dropna().unique().tolist()
        if len(unique) > 2000:
            raise ValueError(f"カテゴリーが多すぎます。ID等でないか確認してください: {column} ({len(unique)})")
        mapping = {value: f"label_{j:05d}" for j, value in enumerate(unique)}
        spec["decode"] = {v: k for k, v in mapping.items()}
        encoded[key] = labels.map(mapping).fillna("missing_category")
        cat_names.append(key)
    schema[column] = spec
if not num_names:
    raise ValueError("このノートブックでは少なくとも1つの可変数値列が必要です。")
# 公式実装のtarget欄に数値列1つを指定します。目的変数を条件固定せず全列を同時生成します。
target_key = num_names[0]
metadata = {"name": DATA_NAME, "task_type": "regression", "header": "infer",
    "column_names": encoded.columns.tolist(), "file_type": "csv",
    "num_col_idx": [encoded.columns.get_loc(c) for c in num_names if c != target_key],
    "cat_col_idx": [encoded.columns.get_loc(c) for c in cat_names],
    "target_col_idx": [encoded.columns.get_loc(target_key)],
    "data_path": f"data/{DATA_NAME}/{DATA_NAME}.csv",
    "test_path": f"data/{DATA_NAME}/holdout.csv", "val_path": None}
DATA_DIR = TABDIFF_REPO / "data" / DATA_NAME
DATA_DIR.mkdir(parents=True, exist_ok=False)
(TABDIFF_REPO / "data/Info").mkdir(parents=True, exist_ok=True)
encoded.iloc[train_idx].to_csv(DATA_DIR / f"{DATA_NAME}.csv", index=False)
encoded.iloc[test_idx].to_csv(DATA_DIR / "holdout.csv", index=False)
(TABDIFF_REPO / "data/Info" / f"{DATA_NAME}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
pd.to_pickle({"schema": schema, "constants": constants, "columns": real.columns.tolist(),
              "train_idx": train_idx, "test_idx": test_idx, "dtypes": real.dtypes.to_dict()}, OUT / "preprocessing.pkl")
display(pd.DataFrame({"列名": list(schema), "変換": [s["kind"] for s in schema.values()]}))
print("学習:",len(train_idx),"ホールドアウト:",len(test_idx),"モデル入力列数:",encoded.shape[1])

In [ ]:
# 公式スクリプトでNumPy配列とinfo.jsonを生成します。
run_command([TABDIFF_PYTHON, "process_dataset.py", "--dataname", DATA_NAME], "preprocess.log")
# 同じ前処理情報・学習/ホールドアウト表の組み合わせを記録します。
training_assets = {
    "preprocessing_sha256": sha256_file(OUT / "preprocessing.pkl"),
    "train_csv_sha256": sha256_file(DATA_DIR / f"{DATA_NAME}.csv"),
    "holdout_csv_sha256": sha256_file(DATA_DIR / "holdout.csv"),
    "ema_update": "per_batch", "ema_decay": EMA_DECAY,
    "dataname": DATA_NAME, "exp_name": EXP_NAME,
}
(OUT / "training_assets.json").write_text(json.dumps(training_assets, indent=2), encoding="utf-8")


## モデルの学習・保存
EMAはバッチ学習後に更新します（減衰率0.997）。学習終了時に通常重みmodel_final_200.ptとEMA重みfinal_ema_200.ptを保存します。epoch数でファイル名末尾は変わります。
画面には経過時間・予想総所要時間のみ表示します。詳細はtrain.logへ保存。学習途中で中断すると最終重みは保存されません。


In [ ]:
# 専用ラッパー内で互換処理と時間計測を行います。公式ソースは変更しません。
runner_source = "\nimport os, sys, json, argparse, importlib, time\nfrom pathlib import Path\nimport torch\ncfg = json.loads(Path(sys.argv[1]).read_text())\nsys.path.insert(0, cfg['repo'])\nos.chdir(cfg['repo'])\n# tqdmをクラスとして保持したまま無効化し、関連ライブラリとの互換性も保ちます。\nimport tqdm\n_OriginalTqdm = tqdm.tqdm\nclass QuietTqdm(_OriginalTqdm):\n    def __init__(self, *args, **kwargs):\n        kwargs['disable'] = True\n        super().__init__(*args, **kwargs)\ntqdm.tqdm = QuietTqdm\nmodule = importlib.import_module('tabdiff.main')\n# PyTorchの版による引数の違いを吸収します（公式ファイルは変更しません）。\n# verbose非対応の版でのみ引数を除去し、他の引数はそのまま渡します。\nimport inspect\ntrainer_module = importlib.import_module('tabdiff.trainer')\n_original_plateau = trainer_module.ReduceLROnPlateau\nif 'verbose' not in inspect.signature(_original_plateau).parameters:\n    def compatible_plateau(*args, **kwargs):\n        kwargs.pop('verbose', None)\n        return _original_plateau(*args, **kwargs)\n    trainer_module.ReduceLROnPlateau = compatible_plateau\n    print('PyTorch compatibility: ReduceLROnPlateau verbose argument omitted')\n\nTrainer = module.Trainer\n\n# バッチ学習の直後にEMA更新。公式側のエポック末更新は無効化します。\n_original_update_ema = trainer_module.update_ema\n_original_run_step = Trainer._run_step\n\ndef run_step_with_batch_ema(self, *args, **kwargs):\n    result = _original_run_step(self, *args, **kwargs)\n    _original_update_ema(self.ema_model.parameters(),\n                         self.diffusion._denoise_fn.parameters(), rate=self.ema_decay)\n    _original_update_ema(self.ema_num_schedule.parameters(),\n                         self.diffusion.num_schedule.parameters(), rate=self.ema_decay)\n    _original_update_ema(self.ema_cat_schedule.parameters(),\n                         self.diffusion.cat_schedule.parameters(), rate=self.ema_decay)\n    return result\n\nTrainer._run_step = run_step_with_batch_ema\n\ndef skip_epoch_ema(*args, **kwargs):\n    pass\ntrainer_module.update_ema = skip_epoch_ema\n\noriginal_load = module.src.load_config\noriginal_loop = Trainer.run_loop\n\ndef configured_load(path):\n    value = original_load(path)\n    value['train']['main'].update(steps=cfg['epochs'], batch_size=cfg['batch'],\n                                check_val_every=cfg['epochs'] + 1, ema_decay=cfg['ema_decay'])\n    value['sample']['batch_size'] = cfg['sample_batch']\n    return value\nmodule.src.load_config = configured_load\n\nclass TimingLogger:\n    def __init__(self, logger, started):\n        self.logger, self.started = logger, started\n    def __getattr__(self, name):\n        return getattr(self.logger, name)\n    def log(self, data, *args, **kwargs):\n        result = self.logger.log(data, *args, **kwargs)\n        if 'epoch' in data:\n            progress = {'completed_epochs': int(data['epoch']),\n                        'training_elapsed': time.monotonic()-self.started}\n            target = Path(cfg['progress_path'])\n            temp = target.with_suffix('.tmp')\n            temp.write_text(json.dumps(progress), encoding='utf-8')\n            os.replace(temp, target)\n        return result\n\ndef train_and_save(self):\n    self.logger = TimingLogger(self.logger, time.monotonic())\n    original_loop(self)\n    save_dir = Path(self.model_save_path)\n    normal_path = save_dir / ('model_final_%d.pt' % self.curr_epoch)\n    ema_path = save_dir / ('final_ema_%d.pt' % self.curr_epoch)\n    # 書き込み途中のファイルをモデル一覧に載せないよう、一時ファイルから置換。\n    states = [\n        (normal_path, {'denoise_fn': self.diffusion._denoise_fn.state_dict(),\n                      'num_schedule': self.diffusion.num_schedule.state_dict(),\n                      'cat_schedule': self.diffusion.cat_schedule.state_dict()}),\n        (ema_path, {'denoise_fn': self.ema_model.state_dict(),\n                   'num_schedule': self.ema_num_schedule.state_dict(),\n                   'cat_schedule': self.ema_cat_schedule.state_dict()}),\n    ]\n    for path, weights in states:\n        temp = path.with_suffix('.tmp')\n        torch.save(weights, temp)\n        os.replace(temp, path)\n    Path(cfg['checkpoint_manifest']).write_text(str(ema_path.resolve()), encoding='utf-8')\n    summary = {'normal': str(normal_path.resolve()), 'ema': str(ema_path.resolve()),\n               'epochs': self.curr_epoch, 'ema_update': 'per_batch', 'ema_decay': self.ema_decay}\n    Path(cfg['models_manifest']).write_text(json.dumps(summary, indent=2), encoding='utf-8')\n    print('Normal model checkpoint:', normal_path)\n    print('EMA model checkpoint:', ema_path)\nTrainer.run_loop = train_and_save\n\nmode = sys.argv[2]\nif mode != 'train':\n    raise RuntimeError('18 is training-only. Use notebook 19 to generate synthetic data.')\nif cfg['gpu'] >= 0 and not torch.cuda.is_available():\n    raise RuntimeError('CUDA unavailable. Set GPU=-1 for CPU, or fix CUDA environment.')\nargs = argparse.Namespace(dataname=cfg['dataname'], mode=mode, method='tabdiff',\n    gpu=cfg['gpu'], device=('cuda:%d'%cfg['gpu']) if cfg['gpu']>=0 else 'cpu',\n    debug=False, no_wandb=True, exp_name=cfg['exp_name'], deterministic=True,\n    y_only=False, non_learnable_schedule=False, num_samples_to_generate=cfg['n_generate'],\n    ckpt_path=None, report=False, num_runs=1, impute=False)\nmodule.main(args)\n"
RUNNER = OUT / "run_tabdiff.py"
RUNNER.write_text(runner_source, encoding="utf-8")
config = {"repo": str(TABDIFF_REPO), "dataname": DATA_NAME, "exp_name": EXP_NAME,
          "epochs": EPOCHS, "batch": TRAIN_BATCH_SIZE, "ema_decay": EMA_DECAY, "ema_update": "per_batch",
          "sample_batch": DEFAULT_SAMPLE_BATCH_SIZE,
          "gpu": GPU, "n_generate": DEFAULT_GENERATE_COUNT, "commit": commit,
          "checkpoint_manifest": str(OUT / "checkpoint.txt"),
          "progress_path": str(OUT / "training_progress.json"),
          "models_manifest": str(OUT / "models.json")}
CONFIG = OUT / "run_config.json"
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")


In [ ]:
# 学習のみ実行します。画面には経過時間と予想総所要時間だけを表示します。
run_command([TABDIFF_PYTHON, RUNNER, CONFIG, "train"], "train.log", show_timing=True)


## 次の操作
19で同じ学習実行の「通常モデル」と「EMAモデル」をそれぞれ選び、同じ生成件数・乱数設定で生成します。20で結果を比較してください。
モデルとともに、artifacts/18_tabdiffの前処理情報とexternal/TabDiff/data内の対応データを保持してください。モデルだけでは19・20は実行できません。
